# Scat Feature Extraction Laboratory

Local notebook for inspecting voice/scat recordings before designing the final real-time scat-to-controls extractor.

Main outputs on a 16 ms grid:

- `f0(t)`: TorchCREPE Hz
- `loudness(t)`: frame RMS z-score
- `gate(t)`: fused causal activity gate, not Silero-only
- `offset(t)`: falling edge of fused gate
- `note_age(t)`: causal counter reset by accepted note onset
- `periodicity(t)`: TorchCREPE confidence
- `onset_strength(t)`: boundary/consonant evidence near accepted note onsets
- `articulation_id(t)`: provisional note-latched lab heuristic

Put recordings in `/workspace/learn/voice_inputs/`, or set `AUDIO_PATH` below.

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path('/workspace')
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from vocal_controls import (
    VocalControlConfig,
    ensure_notebook_dependencies,
    list_audio_inputs,
    display_upload_widget,
    load_selected_audio,
    extract_voice_controls,
    print_summary,
    plot_voice_control_dashboard,
    export_voice_control_features,
)

CONFIG = VocalControlConfig(workspace_root=WORKSPACE)
CONFIG.torchcrepe_device = 'cuda' if False else CONFIG.torchcrepe_device

print('workspace:', CONFIG.workspace_root)
print('voice input directory:', CONFIG.input_dir)
print('ContentVec checkpoint:', CONFIG.contentvec_checkpoint)
print('available recordings:')
for p in list_audio_inputs(CONFIG.input_dir)[:10]:
    print(' -', p)

## Optional Local Upload UI

This works only when the current Jupyter front-end supports `ipywidgets`. In SSH/dev-container workflows, the reliable path is usually to copy audio into `/workspace/learn/voice_inputs/` and rerun the loader cell.

In [ ]:
SHOW_UPLOAD_WIDGET = False

if SHOW_UPLOAD_WIDGET:
    display_upload_widget(CONFIG.input_dir)

## Load And Extract Controls

Set `AUDIO_PATH` to a specific recording, or leave it as `None` to use the newest supported audio file in `/workspace/learn/voice_inputs/`.

In [ ]:
AUDIO_PATH = None  # Example: '/workspace/learn/voice_inputs/Scat 1.m4a'
INSTALL_MISSING_PACKAGES = False

if INSTALL_MISSING_PACKAGES:
    ensure_notebook_dependencies(CONFIG)

audio = load_selected_audio(CONFIG, AUDIO_PATH)
print(audio['source_reason'])
print(audio['source_name'])
print('samples:', len(audio['y']), 'sr:', audio['sr'])

result = extract_voice_controls(audio, CONFIG)
print_summary(result)

## Debug Dashboard

The final gate is the fused activity gate. Silero appears only as diagnostic evidence, so quiet or unvoiced/percussive scat can still remain active if relative energy, high-frequency/ZCR/HPSS evidence, or periodicity supports it.

Plot conventions:

- green vertical lines: accepted note onsets
- purple vertical lines: fused gate offsets
- orange dotted lines: raw spectral-flux candidates
- blue/Silero tracks: diagnostics, not the final gate owner

In [ ]:
fig = plot_voice_control_dashboard(result)

## Inspect Control Tensors

In [ ]:
import pandas as pd

control_df = pd.DataFrame(result['control_tensor'], columns=result['control_names'])
control_df.insert(0, 'time_seconds', result['times'])
control_df['articulation_name'] = result['articulation_name']

print('control tensor:', result['control_tensor'].shape)
print('feature tensor plus:', result['feature_tensor_plus'].shape)
print('control names:')
for name in result['control_names']:
    print(' -', name)

control_df.head(20)

## Export Current Analysis

Exports are useful when comparing different recordings or tuning thresholds outside the notebook.

In [ ]:
EXPORT = False
EXPORT_DIR = WORKSPACE / 'learn' / 'voice_control_exports'

if EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(str(result['source_name'])).stem or 'scat'
    out_prefix = EXPORT_DIR / f'{stem}_16ms_controls'
    out_npz, out_csv, exported_df = export_voice_control_features(result, out_prefix)
    print('wrote:', out_npz)
    print('wrote:', out_csv)
    display(exported_df.head())

## Tuning Notes

Useful knobs live in `VocalControlConfig`:

- `fused_gate_open_threshold` / `fused_gate_close_threshold`: final activity hysteresis thresholds.
- `energy_margin_db`: relative threshold above the recording noise floor.
- `silero_activity_weight`: contribution of Silero evidence to the fused activity score.
- `onset_internal_height` / `onset_internal_prominence`: internal note boundary strictness.
- `onset_min_distance_seconds`: minimum distance between accepted onsets.

For quiet vocal notes, first lower `energy_margin_db` or fused gate thresholds. For false positives in breath/noise, raise `fused_gate_open_threshold` or lower `unvoiced/percussive` feature weights in `vocal_controls.py`.